## Imports

In [1]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../'))

from neuro_fuzzy_toolbox import ANFIS, Hybrid_learning_algorithm, EarlyStopping, get_measures

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Multiclass Classification

## Data

In [4]:
from ucimlrepo import fetch_ucirepo

In [5]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

In [6]:
X = X.fillna(value=0)

In [7]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [8]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[115  38  25  25   9]


In [9]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[49 17 11 10  4]


In [10]:
scaler = MinMaxScaler(feature_range=(0, 1))

x_train = scaler.fit_transform(x_train)

x_test = scaler.transform(x_test)

In [11]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [12]:
y_train.dtype

torch.int64

In [13]:
x_train = x_train.type(torch.float32)
x_test = x_test.type(torch.float32)

In [14]:
loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 8, shuffle = True)
x_trainset = loader.dataset.tensors[0]
y_trainset = loader.dataset.tensors[1]

In [15]:
y_trainset.dtype

torch.int64

## Model & Training

In [16]:
model = ANFIS(
    input_size = 13,
    fuzzy_rules = 6,
    outputs = 5,
    rule_reduced = True,
    output_type='multiclass',
    dtype=torch.float32
)

In [17]:
loss_fn = nn.functional.cross_entropy

optimizer = torch.optim.AdamW
params = {'lr': 0.01, 'weight_decay': 0.1}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = EarlyStopping(
    patience=50, 
    delta=0.01
)

In [18]:
trainer = Hybrid_learning_algorithm(
    epochs=500,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    validation=0.3,
    early_stopping=early_stopping
)

In [19]:
trainer(model, loader, verbose=True)

Epoch:   1/500 - loss: 1.348067 - validation loss: 1.426941
Epoch:   2/500 - loss: 1.320823 - validation loss: 1.346568
Epoch:   3/500 - loss: 1.281032 - validation loss: 1.284844
Epoch:   4/500 - loss: 1.296008 - validation loss: 1.306637
Epoch:   5/500 - loss: 1.312547 - validation loss: 1.295376
Epoch:   6/500 - loss: 1.335486 - validation loss: 1.341848
Epoch:   7/500 - loss: 1.326431 - validation loss: 1.330699
Epoch:   8/500 - loss: 1.334925 - validation loss: 1.344037
Epoch:   9/500 - loss: 1.353981 - validation loss: 1.357301
Epoch:  10/500 - loss: 1.340100 - validation loss: 1.324840
Epoch:  11/500 - loss: 1.354040 - validation loss: 1.328630
Epoch:  12/500 - loss: 1.352652 - validation loss: 1.310866
Epoch:  13/500 - loss: 1.384307 - validation loss: 1.343452
Epoch:  14/500 - loss: 1.452249 - validation loss: 1.428394
Epoch:  15/500 - loss: 1.503654 - validation loss: 1.486241
Epoch:  16/500 - loss: 1.518364 - validation loss: 1.514263
Epoch:  17/500 - loss: 1.475297 - valida

In [ ]:
train_measures = get_measures(model, x_trainset, y_trainset)
for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.6273584905660378
Precision: 0.6080797778910987
Recall: 0.6273584905660378
F1: 0.5892404383257049
Confusion Matrix: [[104   3   5   3   0]
 [ 24   7   5   2   0]
 [  5   2  15   3   0]
 [  4   4  11   6   0]
 [  1   2   1   4   1]]


In [21]:
test_measures = get_measures(model, x_test, y_test)
for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.5604395604395604
Precision: 0.4999632551356689
Recall: 0.5604395604395604
F1: 0.525573669765926
Confusion Matrix: [[44  2  1  2  0]
 [10  2  2  3  0]
 [ 3  1  4  3  0]
 [ 1  5  3  1  0]
 [ 0  1  0  3  0]]
